In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp04/ex_6.py ──
"""
Shared infrastructure for MLFP04 Exercise 6 — NLP & Topic Modelling.

Contains: corpus loading, TF-IDF/CountVectorizer construction, NPMI
coherence scoring, sentiment lexicons, Singapore/APAC text-analytics
scenario helpers, and plot output directory management.

Technique-specific code (TF-IDF/BM25 scoring, NMF decomposition, LDA
fitting, BERTopic pipeline, Word2Vec sentiment classifier) does NOT
belong here — it lives in the per-technique files under
modules/mlfp04/solutions/ex_6/.
"""

from collections import Counter
from pathlib import Path
from typing import Iterable

import numpy as np
import polars as pl


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "ex6_nlp"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# TOY CORPUS — small Singapore-flavoured corpus for teaching derivations
# ════════════════════════════════════════════════════════════════════════

TOY_CORPUS: list[str] = [
    "Singapore economy grew strongly in 2024",
    "Singapore property market shows resilience",
    "MAS tightens monetary policy amid global uncertainty",
    "Property developers report strong demand",
    "Singapore government announces new housing measures",
    "Global markets react to central bank decisions",
    "Technology sector leads Singapore stock exchange",
    "Housing prices continue upward trend in Singapore",
]


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore/APAC document corpus
# ════════════════════════════════════════════════════════════════════════


def load_corpus(min_chars: int = 100) -> pl.DataFrame:
    """Load the MLFP document corpus (title + content + category).

    Returns a polars DataFrame with a derived 'text' column that concatenates
    title and content, filtered to rows with content longer than ``min_chars``.
    Categories include singapore_economy, singapore_housing, asean_economy,
    ml_fundamentals, and several other topical clusters — well-suited to
    unsupervised topic discovery.
    """
    loader = MLFPDataLoader()
    df = loader.load("mlfp03", "documents.parquet")
    return df.with_columns(
        (pl.col("title") + ". " + pl.col("content")).alias("text"),
    ).filter(pl.col("content").str.len_chars() > min_chars)


def corpus_as_lists(
    df: pl.DataFrame,
) -> tuple[list[str], list[str]]:
    """Split a corpus frame into parallel (documents, categories) lists."""
    return df["text"].to_list(), df["category"].to_list()


# ════════════════════════════════════════════════════════════════════════
# NPMI TOPIC COHERENCE
# ════════════════════════════════════════════════════════════════════════


def compute_npmi(
    documents: list[str],
    topic_words: list[list[str]],
    max_docs: int = 3000,
) -> list[float]:
    """Compute Normalised Pointwise Mutual Information coherence per topic.

    NPMI(w_i, w_j) = log(P(w_i, w_j) / (P(w_i) P(w_j))) / (-log P(w_i, w_j))

    Range is [-1, 1]. Higher = more coherent topic. We approximate
    probabilities from a document-level co-occurrence count over the first
    ``max_docs`` documents (enough for a stable signal).
    """
    docs = documents[:max_docs]
    word_doc_count: Counter[str] = Counter()
    pair_doc_count: Counter[tuple[str, str]] = Counter()
    n_docs = len(docs)

    for doc in docs:
        words = set(doc.lower().split())
        for w in words:
            word_doc_count[w] += 1
        word_list = list(words)
        for i in range(len(word_list)):
            for j in range(i + 1, len(word_list)):
                pair = tuple(sorted([word_list[i], word_list[j]]))
                pair_doc_count[pair] += 1

    coherences: list[float] = []
    for topic in topic_words:
        npmi_sum = 0.0
        n_pairs = 0
        for i in range(len(topic)):
            for j in range(i + 1, len(topic)):
                w_i, w_j = topic[i].lower(), topic[j].lower()
                pair = tuple(sorted([w_i, w_j]))
                p_i = word_doc_count.get(w_i, 0) / max(n_docs, 1)
                p_j = word_doc_count.get(w_j, 0) / max(n_docs, 1)
                p_ij = pair_doc_count.get(pair, 0) / max(n_docs, 1)
                if p_ij > 0 and p_i > 0 and p_j > 0:
                    pmi = np.log(p_ij / (p_i * p_j))
                    npmi = pmi / (-np.log(p_ij))
                    npmi_sum += npmi
                    n_pairs += 1
        coherences.append(npmi_sum / max(n_pairs, 1))
    return coherences


# ════════════════════════════════════════════════════════════════════════
# SENTIMENT LEXICONS — baseline for review triage
# ════════════════════════════════════════════════════════════════════════

POSITIVE_WORDS: frozenset[str] = frozenset(
    {
        "good",
        "great",
        "excellent",
        "best",
        "strong",
        "growth",
        "resilience",
        "success",
        "positive",
        "improved",
        "innovative",
        "robust",
        "outperformed",
        "gains",
        "upbeat",
    }
)

NEGATIVE_WORDS: frozenset[str] = frozenset(
    {
        "bad",
        "poor",
        "decline",
        "fall",
        "weak",
        "crisis",
        "risk",
        "loss",
        "negative",
        "failed",
        "uncertainty",
        "downturn",
        "slump",
        "recession",
        "worry",
    }
)


def lexicon_sentiment(docs: Iterable[str]) -> np.ndarray:
    """Score each document in ``docs`` as a scalar in [-1, 1] via a naive lexicon.

    (pos - neg) / (pos + neg), zero when neither list matches.
    """
    scores: list[float] = []
    for doc in docs:
        words = set(doc.lower().split())
        pos = len(words & POSITIVE_WORDS)
        neg = len(words & NEGATIVE_WORDS)
        total = pos + neg
        scores.append((pos - neg) / total if total > 0 else 0.0)
    return np.asarray(scores, dtype=np.float64)


# ════════════════════════════════════════════════════════════════════════
# SCENARIO HELPERS — Singapore / APAC text analytics context
# ════════════════════════════════════════════════════════════════════════

SCENARIOS: dict[str, str] = {
    "tfidf_bm25": (
        "CASE: ST Engineering internal document search. 180K engineering "
        "reports across aerospace, marine, and electronics business units. "
        "TF-IDF vs BM25 determines which report a manager sees first when "
        "searching 'turbine blade fatigue.' A missed report costs ~S$450K "
        "in duplicated R&D work. BM25's length normalisation (b=0.75) "
        "ensures short executive summaries compete fairly with 80-page "
        "technical manuals, improving first-hit accuracy by ~14%."
    ),
    "nmf_topics": (
        "CASE: Singapore Press Holdings (SPH) newsroom content tagging. "
        "~2,400 articles/day across The Straits Times, Business Times, "
        "zaobao.com. NMF runs nightly on the 24-hour TF-IDF matrix and "
        "produces 20 interpretable topics (housing, MAS monetary policy, "
        "SEA Games, etc.) for the recommendation engine. Non-negativity "
        "means the topic-keyword report is directly auditable by the "
        "editorial desk. Ad yield on tagged articles is +11% vs untagged."
    ),
    "lda_topics": (
        "CASE: Monetary Authority of Singapore (MAS) enforcement scanning. "
        "~60K complaint emails/year across retail banking, insurance, "
        "fintech. LDA's mixed-membership model matters — a single complaint "
        "about a crypto rug pull mentioning SGD withdrawals touches BOTH "
        "a 'digital asset fraud' topic AND a 'cross-border payments' topic. "
        "Soft topic assignments route the complaint to both enforcement "
        "teams, cutting median resolution time from 18 days to 11."
    ),
    "bertopic": (
        "CASE: Grab customer-support ticket clustering across Singapore, "
        "Indonesia, Thailand, Vietnam. ~35K tickets/week, mixed English + "
        "Bahasa Indonesia + Thai + Vietnamese. BERTopic's multilingual "
        "sentence-transformer embeddings discover topics that TF-IDF "
        "misses ('driver helmet policy' clusters across languages because "
        "the embeddings are language-agnostic). NPMI coherence of 0.18 vs "
        "LDA's 0.08; each well-clustered ticket saves ~S$3.20 in routing."
    ),
    "sentiment_word2vec": (
        "CASE: DBS Bank app-store review triage. 40K reviews/month across "
        "iOS, Google Play, and Huawei AppGallery in English, Mandarin, "
        "Malay, Tamil. A Word2Vec + logistic-regression sentiment classifier "
        "trained on labelled English reviews transfers to the other three "
        "languages via shared subword tokens. Each negative review routed "
        "to CX within 10 min prevents ~S$8K of avoided viral complaints — "
        "catching 20 extra/month is S$160K vs S$120 in compute."
    ),
}


def print_scenario(name: str) -> None:
    """Print a named Singapore/APAC scenario block for the APPLY phase."""
    body = SCENARIOS.get(name, "")
    if not body:
        return
    print("\n" + "=" * 70)
    print(f"  APPLY — {name}")
    print("=" * 70)
    print(body)
    print("=" * 70 + "\n")




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 6.4: BERTopic — Neural Topic Modelling
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Build a BERTopic pipeline: sentence embeddings -> UMAP -> HDBSCAN
#     -> c-TF-IDF topic extraction
#   - Compare NPMI coherence against NMF/LDA
#   - Fall back to NMF when sentence-transformers is not installed
#   - Apply BERTopic to Grab multilingual support-ticket clustering
#
# PREREQUISITES: Ex 6.2 (NMF), Ex 6.3 (LDA), Ex 3 (UMAP), Ex 1 (HDBSCAN).
# ESTIMATED TIME: ~35 min
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer

from kailash_ml import ModelVisualizer


try:
    from bertopic import BERTopic  # type: ignore[import-untyped]
except ImportError:
    BERTopic = None



## TASK 2 + 3 — BUILD and TRAIN


In [ ]:
corpus_df = load_corpus()
documents, categories = corpus_as_lists(corpus_df)
print(f"Corpus: {len(documents):,} documents")

if BERTopic is not None:
    print("\nBERTopic pipeline (embeddings -> UMAP -> HDBSCAN -> c-TF-IDF)")

    # TODO: Construct a BERTopic model with embedding_model="all-MiniLM-L6-v2",
    # min_topic_size=15, nr_topics="auto", verbose=False.
    topic_model = ____

    # TODO: Call fit_transform(documents) and unpack into (topics, probs).
    topics, probs = ____

    topic_info = topic_model.get_topic_info()
    n_topics = int((topic_info["Topic"] >= 0).sum())

    topic_words: list[list[str]] = []
    for topic_id in range(min(n_topics, 10)):
        words = [w for w, _ in topic_model.get_topic(topic_id)[:10]]
        topic_words.append(words)

    print(f"\nTopics discovered: {n_topics}")
    for _, row in topic_info[topic_info["Topic"] >= 0].head(10).iterrows():
        print(f"  Topic {row['Topic']}: {str(row['Name'])[:60]} (n={row['Count']})")

    method_label = "BERTopic"
    doc_topics_vec = np.asarray(topics)
else:
    # Fallback: NMF-over-TF-IDF approximation with a loud, actionable message
    print("\n" + "!" * 70)
    print(
        "  bertopic / sentence-transformers not installed — run "
        "`uv sync --extra topic-models` for the real pipeline"
    )
    print("  Running NMF-over-TF-IDF approximation for this session")
    print("!" * 70 + "\n")

    vectorizer = TfidfVectorizer(
        max_features=5000, stop_words="english", max_df=0.95, min_df=5
    )
    tfidf = vectorizer.fit_transform(documents)
    feature_names = vectorizer.get_feature_names_out()

    n_topics = 10
    nmf = NMF(n_components=n_topics, random_state=42, max_iter=400, init="nndsvd")
    W = nmf.fit_transform(tfidf)
    H = nmf.components_
    doc_topics_vec = W.argmax(axis=1)

    topic_words = []
    for t in range(n_topics):
        top_idx = H[t].argsort()[-10:][::-1]
        words = [feature_names[i] for i in top_idx]
        topic_words.append(words)
        count = int((doc_topics_vec == t).sum())
        print(f"  Topic {t}: {', '.join(words[:8])} (n={count})")

    method_label = "NMF-fallback"



### Checkpoint 1


In [ ]:
assert len(topic_words) > 0, "Task 3: should discover at least one topic"
assert all(len(w) >= 5 for w in topic_words), "Task 3: each topic needs 5+ words"
print(
    f"\n[ok] Checkpoint 1 passed — {method_label} produced {len(topic_words)} topics\n"
)



## TASK 4 — VISUALISE


In [ ]:
# TODO: compute NPMI coherence for topic_words over the documents
coherences = ____
mean_npmi = float(np.mean(coherences))
print(f"{method_label} mean NPMI coherence: {mean_npmi:+.4f}")

viz = ModelVisualizer()
coherence_data = {f"Topic_{i}": {"NPMI": float(c)} for i, c in enumerate(coherences)}
fig_coh = viz.metric_comparison(coherence_data)
fig_coh.update_layout(title=f"{method_label} Topic Coherence (NPMI)")
fig_coh.write_html(str(OUTPUT_DIR / "ex6_4_bertopic_coherence.html"))



### Checkpoint 2


In [ ]:
assert len(coherences) == len(topic_words), "Task 4: one NPMI per topic"
assert mean_npmi > -0.5, f"Task 4: mean NPMI > -0.5, got {mean_npmi:.4f}"
print("\n[ok] Checkpoint 2 passed — coherence computed\n")



## TASK 5 — APPLY: Grab Multilingual Ticket Clustering


In [ ]:
print_scenario("bertopic")



## REFLECTION


[x] Assembled BERTopic's embed/reduce/cluster/describe pipeline
  [x] Handled the optional extra with a loud, actionable fallback
  [x] Measured NPMI coherence and compared it to NMF/LDA
  [x] Mapped BERTopic to Grab's multilingual routing problem

  Next: 05_sentiment_word2vec.py — word embeddings as features.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

